In [2]:
%%writefile requirements.txt
numpy==2.3.3
pandas==2.3.3
pmdarima==2.1.1
polars==1.34.0
polars-runtime-32==1.34.0
scikit-learn==1.7.2
statsmodels==0.14.5
xgboost==3.1.1
yfinance==0.2.66

Overwriting requirements.txt


In [3]:
import os

def make_directory(path: str) -> None:
    try:
        os.makedirs(path)
        print(f"created directory '{path}'!")
    except FileExistsError:
        print(f"directory '{path}' already exists!")

make_directory("src")
make_directory("workflows")
make_directory("src/xgboost_model")
make_directory("src/sarimax_model")
make_directory("src/prophet_model")

directory 'src' already exists!
directory 'workflows' already exists!
directory 'src/xgboost_model' already exists!
directory 'src/sarimax_model' already exists!
directory 'src/prophet_model' already exists!


# loading the raw data from `yfinance`

this serves as the base function for loading all data

In [13]:
%%writefile src/load_data.py
"""
function written to easily load and process stock data from `yfinance`.
"""
import pandas as pd
import yfinance as yf
import polars as pl
from datetime import date, datetime 

from src.utils import trading_days_between

def load_stocks(
    stocks: str | list[str], start: str, end: str, use_polars: bool = True
) -> pl.DataFrame | pd.DataFrame:
    """load stock data from `yfinance`.

    stocks are loaded singularly, from start to end date. option to return a 
    polars dataframe or a pandas dataframe.

    Args:
        stocks (list): single-item list of stock tockers.
        start (str): first historical date.
        end (str): last historical date (up to today).
        use_polars (bool, optional): whether to return a polars dataframe or a
            pandas dataframe. defaults to true.

    Raises:
        ValueError: raised if users enter more than 1 ticker

    Returns:
        DataFrame: polars or pandas dataframe, depending on the value of
        `use_polars`.
    """
    if isinstance(stocks, str):
        stocks = [stocks]
        
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")
        
    df = yf.download(stocks, start, end)
    df.index = pd.to_datetime(df.index)
    df.columns = (
        pd.MultiIndex.from_tuples(df.columns) 
        if not isinstance(df.columns, pd.MultiIndex) else df.columns
    )
    df.columns = df.columns.set_names(["Field", "Ticker"])
    df.index.name = "Date"
    df = df.apply(pd.to_numeric, errors="coerce")

    df_out = (
        df.swaplevel("Field", "Ticker", axis=1)
        .sort_index(axis=1)
        .stack("Ticker", future_stack=True)
        .reset_index()
    )

    df_out = df_out.rename(columns=str.lower)

    return pl.from_pandas(df_out) if use_polars else df_out


def get_option_chain(
    ticker: str, expiry: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """return the chain of options available for a given ticker.

    Args:
        ticker (str): ticker symbol to retrieve options for.
        expiry (str): date of expiry.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: list of expiry datasets for calls/puts.
    """
    tk = yf.Ticker(ticker)
    chain = tk.option_chain(expiry)
    calls = chain.calls.copy()
    puts = chain.puts.copy()
    calls["type"] = "call" 
    puts["type"] = "put"
    return calls, puts 


def list_expiries(ticker: str) -> list[str]:
    """simple list of all expiries for a stock ticker."""
    return list(yf.Ticker(ticker).options)


def pick_expiries_for_horizons(
    ticker: str,
    horizons_td: list[int],
    asof: date | None = None 
) -> dict[int, str]:
    """map each horizon to the closest available listed expiry.

    Args:
        ticker (str): ticker symbol to retrieve options for.
        horizons_td (list[int]): list of horizons being forecasted.
        asof (date | None, optional): 'as of' run date. defaults to None.

    Returns:
        dict[int, str]: mapping dictionary of horizons and closest dates of
            expiry.
    """
    if asof is None:
        asof = date.today()
    
    exp_strs = list_expiries(ticker)
    exp_dates = [datetime.strptime(s, "%Y-%m-%d").date() for s in exp_strs]

    mapping = {}
    for h in horizons_td:
        best = None 
        best_dist = None 

        for d, s in zip(exp_dates, exp_strs):
            td = trading_days_between(asof, d)
            dist = abs(td - h)
            if best is None or dist < best_dist:
                best, best_dist = s, dist 
        mapping[h] = best 
    
    return mapping

Overwriting src/load_data.py


# data transformations and feature engineering

## xgboost

In [3]:
%%writefile src/xgboost_model/xgboost_etl.py
"""
set of functions to process `yfinance` data for the XGBoost model.

adds lagged features and time indicators to build the full feature space for
each model.
"""
from __future__ import annotations 

import polars as pl
from datetime import datetime, date, timedelta
import yfinance as yf
from typing import Tuple, Literal

from src.load_data import load_stocks


def prep_columns(df: pl.DataFrame, col: str) -> pl.DataFrame:
    """prep the columns pulled from `yfinance` into a clean dataframe

    adds a 'move' column to indicate overall daily change; time-lapse columns
    lagged over 1, 7, 30 days; and rolling mean and SD values over 7 days.

    Args:
        df (pl.DataFrame): raw `yfinance` dataframe.
        col (str): which column (choose between 'close', 'open') to compute the
            lag features for.

    Returns:
        pl.DataFrame: full dataframe with lagged features of chosen column.
    """
    if col == "move":
        df = df.with_columns(
            (pl.col("close") - pl.col("open")).alias(col)
        )

    df_out =  (
        df.select(
            [
                "date",
                "ticker",
                "volume",
                col
            ]
        )
        .sort([pl.col("ticker"), pl.col("date")], descending=False)
        .with_columns(
            pl.col(col).shift(1).over("ticker").alias(f"prev1_{col}"),
            pl.col(col).shift(7).over("ticker").alias(f"prev7_{col}"),
            pl.col(col).shift(30).over("ticker").alias(f"prev30_{col}"),
        )
        .with_columns(
            pl.col(col)
            .rolling_mean(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_mean_7")
        )
        .with_columns(
            pl.col(col)
            .rolling_std(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_std_7")
        )
    )

    return df_out


def prep_data_frame(df: pl.DataFrame) -> pl.DataFrame:
    """prepare lag columns for all of 'open', 'close', and 'move'.

    Args:
        df (pl.DataFrame): raw `yfinance` stock dataframe.

    Returns:
        pl.DataFrame: processed stock data with lag columns for all price
            indicators.
    """
    markers = ["open", "close", "move"]
    df_out = None

    for marker in markers:
        df_prep = prep_columns(df, marker)
        if df_out is None:
            df_out = df_prep
        else:
            df_out = df_prep.join(
                df_out, on=["date", "ticker", "volume"], how="inner"
            )
    
    return (
        df_out.with_columns(
            pl.col("date").dt.weekday().alias("dow")
        )
        .with_columns(
            pl.col("date").dt.month().alias("month")
        )
        .with_columns(
            pl.when(pl.col("dow").is_in([0, 4]))
            .then(pl.lit(1))
            .otherwise(pl.lit(0))
            .alias("mon_or_fri")
        )
    )

LabelMode = Literal["log_return", "simple_return"]

def build_dataset(
    df: pl.DataFrame, 
    label_col: str = "close", 
    horizon: int = 21, 
    label_mode: LabelMode = "log_return"
) -> pl.DataFrame:
    """wrapper for `prep_data_frame`.

    Args:
        df (pl.DataFrame): raw `yfinance` stock dataframe.
        label (str, optional): which of 'close' or 'move' to process. defaults
            to "close".
        horizon (int, optional): length of forecast horizon. must be >= 1.
        label_mode (Literal): 

    Raises:
        ValueError: only accepts `horizon` values >= 1.
        ValueError: only accepts 'close' or 'move' or 'open'.
        ValueError: only accepts 'log_return' or 'simple_return'.

    Returns:
        pl.DataFrame: fully processed `yfinance` data with nulls removed.
    """
    if label_col not in {"close", "open", "move"}:
        raise ValueError("`label_col` must be one of ['close', 'open', 'move']")
    
    df_feat = prep_data_frame(df)

    if horizon < 1:
        raise ValueError("`horizon` must be >= 1")

    if label_col == "move":
        base = pl.col("move")
    else:
        base = pl.col(label_col)
    
    future = base.shift(-horizon)

    if label_mode == "log_return":
        df_feat = df_feat.with_columns(
            (future.log() - base.log()).alias("label")
        )
    elif label_mode == "simple_return":
        df_feat = df_feat.with_columns(((future / base) - 1.0).alias("label"))
    else:
        raise ValueError(
            "`labl_mode` must be one of ['log_return', 'simple_return']"
        )

    return df_feat.drop_nulls()

Overwriting src/xgboost_model/xgboost_etl.py


## sarimax

In [6]:
%%writefile src/sarimax_model/sarimax_etl.py
"""
set of functions to process `yfinance` data for the SARIMAX model.

pulls code from xgboost model (`build_dataset`) and uses it to add indexes to 
the dataframe. 
"""
import polars as pl
from datetime import datetime, date, timedelta
import yfinance as yf
from typing import Tuple

from src.load_data import load_stocks
from src.xgboost_model.xgboost_etl import build_dataset


def build_df_with_indices(
    df: pl.DataFrame, label: str, start: str, end: str
) -> pl.DataFrame:
    """process raw dataframe and add indices.

    this is originally intended to bolster the SARIMAX model by adding broad
    exogenous variables to the features space.

    Args:
        df (pl.DataFrame): raw `yfiance` stock dataframe.
        label (str): 'close' or 'move' price indicator.
        start (str): start date for index pulling from `yfinance`.
        end (str): end date for index pulling from `yfinance`.

    Returns:
        pl.DataFrame: features data with lagged columns and added index values.
    """
    indices_list = ["SPY", "QQQ", "IWM", "VXX", "UUP", "HYG", "LQD"]
    df_idx_out = None

    for idx in indices_list:
        idx_cl = idx.replace("^", "")

        df_idx = load_stocks([idx], start, end).select(
            pl.col("date"),
            pl.col("close").alias(f"{idx_cl}_close"),
            pl.col("volume").alias(f"{idx_cl}_volume")
        )

        if df_idx_out is None:
            df_idx_out = df_idx
        else:
            df_idx_out = df_idx_out.join(df_idx, on=["date"], how="inner")
    
    df_ticker = build_dataset(df, label)

    df_out = df_ticker.join(df_idx_out, on=["date"], how="inner")
    
    return df_out

Overwriting src/sarimax_model/sarimax_etl.py


## prophet

In [7]:
%%writefile src/prophet_model/prophet_etl.py
"""
set of functions to process `yfinance` data for the Prophet model.

focuses on pulling indexes and computing returns and volatility to add these as
regressors in Prophet.
"""
import yfinance as yf 
import polars as pl 
import pandas as pd
from typing import Tuple

from src.load_data import load_stocks 


class GetSectorETF:
    """simple class to infer sector ETFs"""
    def __init__(
        self,
        indexes: list,
        label: str,
        target_ticker: str
    ):
        self.SECTOR_TO_ETF = {
            "Technology": "XLK",
            "Communication Services": "XLC",
            "Financial Services": "XLF",
            "Energy": "XLE",
            "Consumer Cyclical": "XLY",
            "Consumer Defensive": "XLP",
            "Industrials": "XLI",
            "Healthcare": "XLV",
            "Real Estate": "XLRE",
            "Utilities": "XLU",
        }

        self.indexes = indexes
        self.target_ticker = target_ticker

        if label not in ["open", "close", "move"]:
            raise ValueError("label must be one of ['open', 'close', 'move]")
        else:
            self.label = label

    def extract_stock_info(self, stock_df: pl.DataFrame) -> Tuple[str, str, str]:
        """extract the ticker, earliest, and latest data from `stock_df`.

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            Tuple[str, str, str]: ticker, start_date, end_date values
        """
        tickers = [stock_df.select("ticker").unique().item()]

        date_df = (
            stock_df
            .select("date")
            .unique()
            .sort(by="date", descending=True)
        )

        min_date = date_df.select("date").tail(1).item().strftime("%Y-%m-%d")
        max_date = date_df.select("date").head(1).item().strftime("%Y-%m-%d")

        return tickers, min_date, max_date

    def infer_sector_etfs(self, stock_df: pl.DataFrame) -> pl.DataFrame:
        """using tickers, extracts the close values of sector ETFs.

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            pl.DataFrame: dataframe of dates, `label` values for relevant ETFs 
                for the stock's ticker, as well as `label` and values for the
                passed-in indexes.
        """
        tickers, start_date, end_date = self.extract_stock_info(stock_df)

        etfs = set()

        for t in tickers:
            info = yf.Ticker(t).info
            sector = info.get("sector")
            if sector in self.SECTOR_TO_ETF:
                etfs.add(self.SECTOR_TO_ETF[sector])
        
        ticker_list = list(etfs)
        ticker_list += self.indexes
        df_out = None

        for ticker in ticker_list:
            df_temp = load_stocks([ticker], start_date, end_date).select(
                pl.col("date"),
                pl.col(self.label).alias(f"{ticker}")
            )

            if df_out is None:
                df_out = df_temp 
            else:
                df_out = df_out.join(df_temp, on=["date"], how="inner")
        
        return df_out
    
    def build_etf_df(self, stock_df: pl.DataFrame) -> pl.DataFrame:
        """combine ETF values with stock dataframe

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            pl.DataFrame: combined dataframe of `stock_df` with ETF values.
        """
        df_etf = self.infer_sector_etfs(stock_df)

        return (
            stock_df.select(
                pl.col("date"), pl.col(self.label).alias(self.target_ticker)
            )
            .join(
                df_etf, on=["date"], how="inner"
            )
            .sort("date")
        )

def compute_returns(df: pl.DataFrame) -> pl.DataFrame:
    """compute the daily return rate.

    Args:
        df (pl.DataFrame): stock dataframe loaded from `yfinance` and processed
            through `GetSectorETF`.

    Returns:
        pl.DataFrame: polars dataframe with daily returns for a target stock and
            desired indexes.
    """
    tickers = [c for c in df.columns if c != "date"]

    for ticker in tickers:
        df = df.with_columns(
            pl.col(ticker).pct_change().alias(f"{ticker}_return")
        )
    
    return df

def compute_return_volatility(df: pl.DataFrame, window: int) -> pd.DataFrame:
    """compute the rolling volatility (SD) of target stock and indexes.

    returns a pandas dataframe for use in prepping data for Prophet.

    Args:
        df (pl.DataFrame): polars dataframe with target stock and index prices
            and returns.
        window (int): rolling window (in days) to compute volatility.

    Returns:
        pd.DataFrame: pandas dataframe with prices, returns, and return 
            volatility.
    """
    tickers = [c for c in df.columns if c.endswith("_return")]

    for ticker in tickers:
        df = df.with_columns(
            pl.col(ticker)
            .rolling_std(window_size=window, min_samples=1)
            .alias(f"{ticker}_rolling_std_{window}")
        )
    
    return df.to_pandas()

def compute_rsi(df: pd.DataFrame, ticker: str, period: int) -> pd.Series:
    """add a column for RSI over a period window.

    Args:
        df (pd.DataFrame): pandas dataframe with target stock PRICES.
        ticker (str): stock to compute RSI for.
        period (int): window (days).

    Returns:
        pd.Series: pandas series to add to `df` as a column containing RSI
            values.
    """
    prices = pd.to_numeric(df[ticker], errors="coerce")
    delta = prices.diff()

    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

def compute_price_volatility(
    df: pd.DataFrame, ticker: str, period: int
) -> pd.Series:
    """add price volatility columns.

    Args:
        df (pd.DataFrame): pandas dataframe with price data.
        ticker (str): stock to compute volatility for.
        period (int): window (days).

    Returns:
        pd.Series: column indicating lagged prices.
    """
    return df[ticker].shift(period)

def build_prophet_df(
    stocks: list,
    start: str,
    end: str,
    label: str,
    indexes: list
) -> pd.DataFrame:
    """run through suite of functions to build the final dataset for Prophet.

    Args:
        stocks (list): _description_
        start (str): _description_
        end (str): _description_
        label (str): _description_
        indexes (list): _description_

    Returns:
        pd.DataFrame: _description_
    """
    tkr = stocks[0]

    df_raw = load_stocks(stocks, start, end)
    etfs = GetSectorETF(indexes=indexes, label=label, target_ticker=tkr)
    df_etf = etfs.build_etf_df(df_raw)

    df_ret = compute_returns(df_etf)
    df_vol = compute_return_volatility(df_ret, 10)

    df_vol["rsi_7"] = compute_rsi(df_vol, tkr, 7)
    df_vol["rsi_14"] = compute_rsi(df_vol, tkr, 14)
    df_vol["rsi_21"] = compute_rsi(df_vol, tkr, 21)
    df_vol["prev7_close"] = compute_price_volatility(df_vol, tkr, 1)
    df_vol["prev14_close"] = compute_price_volatility(df_vol, tkr, 7)
    df_vol["prev30_close"] = compute_price_volatility(df_vol, tkr, 30)

    return (
        df_vol.rename(columns={"date": "ds", f"{tkr}": "y"})
        .dropna()
    )

Overwriting src/prophet_model/prophet_etl.py


# prep the data for modeling

mostly just used for train-test splits but different models call for different  
data parsing (even if only slightly)

In [4]:
%%writefile src/model_preprocess.py
""" 
handle the creation of train-test splits for each model. not all are the same.
split for xgboost is traditional (X,y train/test tables), but for forecasting
models the split is a train/eval split without a test dataframe.

each split is done based on a cutoff date to only allow training on past data 
and testing/eval on future data.
"""
from datetime import datetime
import polars as pl
import pandas as pd
from typing import Tuple

def train_test_split_cutoff(
    df: pl.DataFrame, cutoff: datetime, label_col: str
) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """do a train-test split based on a cutoff value.

    training data is all data prior to the cutoff, testing data is all data on
    or after the cutoff.

    Args:
        df (pl.DataFrame): dataframe to do the split on.
        cutoff (datetime): datetime object indicating the split date.
        label_col (str): label column to indicate which is 'y'.

    Returns:
        Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]: gives the
            "traditional" X_train, X_test, y_train, y_test output (akin to
            sklearn).
    """
    train = df.filter(pl.col("date") < cutoff)
    test = df.filter(pl.col("date") >= cutoff)

    X_train, X_test = (
        train.drop(label_col, "ticker", "date"),
        test.drop(label_col, "ticker", "date")
    )
    y_train, y_test = train[label_col], test[label_col]

    return X_train, X_test, y_train, y_test

def split_ar_on_cutoff(
    df: pl.DataFrame, cutoff: datetime, label: str, exog_feats: list
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """split autoregressive data on a cutoff value.

    this doesn't return unique tables for X and y and is specifically designed
    for SARIMAX. can also be used for training Prophet. a pandas dataframe is
    returned (not polars) for use in the forecasting models.

    Args:
        df (pl.DataFrame): dataframe to do the split on.
        cutoff (datetime): datetime object indicating the split date.
        label (str): label column to indicate which is the endogenous value.
        exog_feats (list): list of exogenous features for SARIMAX.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: train/eval dataframes returned as
            pandas dataframes.
    """
    train = df.filter(pl.col("date") < cutoff)
    eval = df.filter(pl.col("date") >= cutoff)

    train_pd = train.to_pandas()
    eval_pd = eval.to_pandas()

    chg_cols = [f"{label}_rolling_std_7"]

    cols_list = [label] + exog_feats + chg_cols

    train_pd = train_pd[cols_list]
    eval_pd = eval_pd[cols_list]

    return train_pd, eval_pd

def split_prophet_df(
        df: pd.DataFrame, cutoff: datetime
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """train/eval split for Prophet model.

    Args:
        df (pd.DataFrame): pandas dataframe set up for Prophet.
        cutoff (datetime): datetime object indicating the split date.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: train/eval dataframes returned as
            pandas dataframes.
    """
    df_train = df[df["ds"] < cutoff]
    df_eval = df[df["ds"] >= cutoff]

    return df_train, df_eval

Overwriting src/model_preprocess.py


leave this here for now  
trying to mess with `argparse` so it can run in the command line but i'll save  
that for last....

In [9]:
# import argparse

# ### parameters (use argparse module)

# # model hyperparameters
# DEFAULT_NUM_ESTIMATORS = 100
# DEFAULT_LEARNING_RATE = 0.01

# parser = argparse.ArgumentParser(
#     description="model hyperparameters"
# )

# parser.add_argument(
#     "-NUM_ESTIMATORS",
#     type=int,
#     default=DEFAULT_NUM_ESTIMATORS,
#     help="number of estimators"
# )
# parser.add_argument(
#     "-LEARNING_RATE",
#     type=float,
#     default=DEFAULT_LEARNING_RATE,
#     help="how fast the model learns"
# )

# # data parameters
# DEFAULT_LABEL = "move"

# parser.add_argument(
#     "-START_DATE",
#     type=str,
#     default=None,
#     help="data training start date"
# )

# parser.add_argument(
#     "-END_DATE",
#     type=str,
#     default=None,
#     help="data training end date"
# )

# parser.add_argument(
#     "-STOCKS",
#     type=list,
#     default=None,
#     help="stocks to forecast"
# )

# parser.add_argument(
#     "-LABEL",
#     type=str,
#     default=DEFAULT_LABEL,
#     help="one of 'move', 'open', 'close'; which of these values to forecast"
# )

# # cutoff value
# parser.add_argument(
#     "-CUTOFF",
#     type=datetime,
#     default=None,
#     help="cutoff value for train/test split"
# )

# # create args
# args = parser.parse_args()

# NUM_ESTIMATORS = args.NUM_ESTIMATORS
# LEARNING_RATE = args.LEARNING_RATE
# STOCKS = args.STOCKS
# START_DATE = args.START_DATE
# END_DATE = args.END_DATE
# LABEL = args.LABEL
# CUTOFF = args.CUTOFF

# NOW build the model

# training the model

## xgboost

In [5]:
%%writefile src/xgboost_model/train_xgboost_model.py
"""
contains a wrapper function for loading the data and training the XGBoost model.
forecasting is not done here, only model training.
"""
from __future__ import annotations

import polars as pl
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error
from datetime import datetime
from typing import Tuple, Dict, List, Optional

from src.load_data import load_stocks
from src.model_preprocess import train_test_split_cutoff
from src.xgboost_model.xgboost_etl import build_dataset


def _make_base_params(n_estimators: int, learning_rate: float) -> dict:
    return dict(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.005,
        objective="reg:squarederror"
    )


def _fit_point_model(X_train, y_train, params: dict) -> xgb.XGBRegressor:
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    return model 


def _fit_quantile_model(
    X_train, y_train, params: dict, alpha: float
) -> xgb.XGBRegressor:
    qparams = dict(params)
    qparams["objective"] = "reg:quantileerror" 
    qparams["quantile_alpha"] = alpha 
    model = xgb.XGBRegressor(**qparams)
    model.fit(X_train, y_train)
    return model 


def train_xgb_model(
    stocks: List[str],
    start_date: str,
    end_date: str,
    cutoff: datetime,
    label_col: str,
    horizon: int,
    n_estimators: int,
    learning_rate: float,
    quantiles: Optional[List[float]] = None,
    label_mode: str = "log_return"
) -> Tuple[Dict[str, xgb.XGBRegressor], float, pl.DataFrame, List[str]]:
    """load raw data, preprocess, and train XGBoost model.

    trains either a point model (no quantiles) or a quantiles bundle (list of 
    floats).

    Args:
        stocks (list): single-item list of stock tickers.
        start_date (str): when to start the dataframe.
        end_date (str): final date of the dataframe.
        cutoff (datetime): cutoff datetime object for train/test splits.
        label_col (str): label column (y).
        horizon (int): forecast horizon.
        n_estimators (int): XGBoost `n_estimators` hyperparameter.
        learning_rate (float): XGBoost `learning_rate` hyperparameter.
        quantiles (Optional[List[float]]): list of quantiles to train models on.
        label_mode: whether the label is a log or a simple return.

    Raises:
        ValueError: cannot process more than one stock at a time.

    Returns:
        Tuple[Dict[str, xgb.XGBRegressor], float, pl.DataFrame, list]: XGBoost 
        regression model, RMSE value, full dataframe with features, features
        list.
    """
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")

    df_raw = load_stocks(
        stocks=stocks, start=start_date, end=end_date, use_polars=True
    )
    df_feat = build_dataset(
        df=df_raw, label_col=label_col, horizon=horizon, label_mode=label_mode
    )
    X_train, X_test, y_train, y_test = train_test_split_cutoff(
        df=df_feat, cutoff=cutoff, label_col="label"
    )
    feature_cols = X_train.to_pandas().columns.tolist()

    base_params = _make_base_params(
        n_estimators=n_estimators, learning_rate=learning_rate
    )

    models: Dict[str, xgb.XGBRegressor] = {}

    if not quantiles:
        m = _fit_point_model(X_train, y_train, base_params)
        preds = m.predict(X_test)
        rmse = root_mean_squared_error(y_test, preds)
        models["point"] = m 
        return models, rmse, df_feat, feature_cols
    
    for q in quantiles:
        key = f"q{int(round(q*100))}"
        try:
            models[key] = _fit_quantile_model(
                X_train, y_train, base_params, alpha=q
            )
        except TypeError as e:
            raise TypeError(
                "xgboost installed doesn't support sklearn quantile params "
                "('reg:quantileerror' / 'quantile_alpha'). "
                "upgrade xgboost or use the point model + residual bands."
            ) from e 
    
    if "q50" not in models:
        mid = sorted(quantiles)[len(quantiles)//2]
        mid_key = f"q{int(round(mid*100))}"
    else:
        mid_key = "q50" 
    
    preds = models[mid_key].predict(X_test)
    rmse = root_mean_squared_error(y_test, preds)

    return models, rmse, df_feat, feature_cols
        

Overwriting src/xgboost_model/train_xgboost_model.py


# forecasting

this predicts future unseen values (not just test data)

!!!! will need to debug this to align the dates with what sarimax outputs

In [6]:
%%writefile src/xgboost_model/xgboost_forecaster.py
"""
contains a class for forecasting the future value from the trained XGBoost model.
"""
from __future__ import annotations 

import pandas as pd
import polars as pl
import xgboost as xgb
from datetime import timedelta, datetime
from typing import Dict, List 

from src.xgboost_model.xgboost_etl import prep_data_frame
from src.utils import build_trading_future_dates


class XGBExpiryForecaster:
    """
    forecast a horizon-ahead return distribution (or point) from the latest 
    features.
    """
    def __init__(
        self,
        models: Dict[str, xgb.XGBRegressor],
        feature_cols: List[str],
        label_mode: str = "log_return"
    ):
        self.models = models
        self.feature_cols = feature_cols
        self.label_mode = label_mode
    
    def _latest_feature_row(self, df_raw: pl.DataFrame) -> pd.DataFrame:
        df_feat = prep_data_frame(df_raw)
        return df_feat.select(self.feature_cols).tail(1).to_pandas()
    
    def _return_to_price(self, s0: float, r: float) -> float:
        if self.label_mode == "log_return":
            return float(s0 * (2.718281828459045 ** r))
        return float(s0 * (1.0 + r))
    
    def forecast_expiry(
        self,
        df_raw: pl.DataFrame,
        horizon: int,
        price_col: str = "close"
    ) -> pl.DataFrame:
        """forecast to expiry at +horizon trading days
        
        returns 1-row df with date + predicted return quantiles + price quantiles.

        Args:
            df_raw (pl.DataFrame): raw `yfinance` stock data.
            horizon (int): forecast horizon trading days.
            price_col (str, optional): the initial column off of which the label
                was built. defaults to 'close'. 
        
        Returns:
            pl.DataFrame: table with dates and predicted values.
        """
        last_date = df_raw["date"][-1]
        
        if isinstance(last_date, datetime):
            last_dt = last_date
        else:
            last_dt = datetime.combine(last_date, datetime.min.time())
        
        expiry_date = build_trading_future_dates(last_dt, horizon)[-1]

        s0 = float(df_raw[price_col][-1])
        X = self._latest_feature_row(df_raw)

        out = {"date": [expiry_date], "spot": [s0], "horizon": [horizon]}

        for name, m in self.models.items():
            rhat = float(m.predict(X)[0])
            out[f"pred_{name}_ret"] = [rhat]
            out[f"pred_{name}_px"] = [self._return_to_price(s0, rhat)]
        
        return pl.from_pandas(pd.DataFrame(out))


class XGBStockForecaster:
    """
    use the trained XGBoost model to make a forecast over a specified interval.
    """
    def __init__(
        self, model: xgb.XGBRegressor, feature_cols: list, label_col: str
    ):
        self.model = model
        self.feature_cols = feature_cols
        self.label_col = label_col
    
    def predict_one(self, df_raw: pl.DataFrame) -> float:
        """predict using the most recent row of features built from `df_raw`"""
        df_feat = prep_data_frame(df_raw)
        row_pd = df_feat.select(self.feature_cols).tail(1).to_pandas()
        pred = self.model.predict(row_pd)
        return float(pred[0])

Overwriting src/xgboost_model/xgboost_forecaster.py


# full modeling pipeline

raw data -> ETL -> split -> train a model -> evaluate model training -> forecast

In [2]:
%%writefile src/xgboost_model/xgboost_pipeline.py
"""
full pipeline for loading, transforming, training, and forecasting data for the
XGBoost model.
"""
from __future__ import annotations 

import pandas as pd
import polars as pl
from datetime import datetime
from typing import Tuple, Dict, List, Optional 
import numpy as np

from src.load_data import load_stocks
from src.utils import build_forecast_dates
from src.xgboost_model.train_xgboost_model import train_xgb_model
from src.xgboost_model.xgboost_forecaster import XGBStockForecaster, XGBExpiryForecaster

def train_and_forecast_xgb_options(
    ticker: str,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    label_col: str = "close",
    horizons: List[int] = [10, 21, 40],
    quantiles: Optional[List[float]] = [0.1, 0.5, 0.9],
    label_mode: str = "log_return",
    n_estimators: int = 400,
    learning_rate: float = 0.03,
) -> Tuple[pl.DataFrame, Dict[int, float], Dict[int, float]]:
    """full pipeline for forecasting options.

    for each horizon, train model(s) and forecast expiry distribution. returns
    combined forecast table (rows = horizons) and RMSEs per horizon.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        cutoff (datetime): datetime object for train-test split.
        label_col (str, optional): which value to predict. defaults to "close".
        horizons (List[int], optional): horizon days to forecast. 
            defaults to [10, 21, 40].
        quantiles (Optional[List[float]], optional): quantile distributions to
            model on. defaults to [0.1, 0.5, 0.9].
        label_mode (str, optional): how to transform the label. defaults to
            "log_return".
        n_estimators (int, optional): XGBoost `n_estimators` hyperparameter.
            defaults to 400.
        learning_rate (float, optional): XGBoost `learning_rate` hyperparameter. 
            defaults to 0.03.

    Returns:
        Tuple[pl.DataFrame, Dict[int, float], Dict[int, float]]: dataframe of p
            redicted values per horizon and the RMSE from training and the 
            improvements.
    """
    stocks = [ticker]
    df_raw = load_stocks(stocks, start_date, end_date, use_polars=True)

    forecasts = []
    rmses: Dict[int, float] = {}
    improvements: Dict[int, float] = {}

    for h in horizons:
        models, rmse, df_feat, feature_cols = train_xgb_model(
            stocks=stocks,
            start_date=start_date,
            end_date=end_date,
            cutoff=cutoff,
            label_col=label_col,
            horizon=h,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            quantiles=quantiles,
            label_mode=label_mode
        )
        rmses[h] = rmse 
        
        y_test = (
            df_feat
            .filter(pl.col("date") >= cutoff)
            .select("label")
            .to_numpy()
            .ravel()
        )

        rmse_zero = (
            float(np.sqrt(np.mean(np.square(y_test)))) 
            if len(y_test) 
            else np.nan
        )

        improvement = (
            float(1.0 - (rmse / rmse_zero))
            if np.isfinite(rmse_zero) and rmse_zero > 0
            else np.nan
        )
        improvements[h] = improvement

        forecaster = XGBExpiryForecaster(
            models=models, feature_cols=feature_cols, label_mode=label_mode
        )
        fc = forecaster.forecast_expiry(
            df_raw=df_raw, horizon=h, price_col=label_col
        )
        forecasts.append(fc)
    
    return pl.concat(forecasts, how="vertical"), rmses, improvements


def train_and_forecast_xgb(
    ticker: str,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    horizon_days: int,
    label_col: str = "close",
    n_estimators: int = 200,
    learning_rate: float = 0.05
) -> Tuple[pl.DataFrame, float]:
    """full pipeline for XGBoost model.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.
        label_col (str, optional): which value to predict. defaults to "close".
        n_estimators (int, optional): XGBoost `n_estimators` hyperparameter.
            defaults to 200.
        learning_rate (float, optional): XGBoost `learning_rate` hyperparameter.
            defaults to 0.05.

    Returns:
        Tuple[pl.DataFrame, float]: dataframe of predicted values per date and
            the RMSE from training.
    """
    stocks = [ticker]

    df_raw = load_stocks(stocks, start_date, end_date, use_polars=True)
    last_date = df_raw.select("date").max().item()
    future_dates = build_forecast_dates(last_date, horizon_days)

    preds = []
    rmses = []

    for h in range(1, horizon_days + 1):
        model, rmse, df_feat, feature_cols = train_xgb_model(
            stocks=stocks,
            start_date=start_date,
            end_date=end_date,
            cutoff=cutoff,
            label_col=label_col,
            horizon=h,
            n_estimators=n_estimators,
            learning_rate=learning_rate
        )

        forecaster = XGBStockForecaster(model, feature_cols, label_col=label_col)
        pred = forecaster.predict_one(df_raw)

        preds.append(pred)
        rmses.append(rmse)
    
    rmse_out = float(sum(rmses) / len(rmses))

    df_out = pl.from_pandas(
        pd.DataFrame({"date": future_dates, f"pred_{label_col}": preds})
    )

    return df_out, rmse_out

Overwriting src/xgboost_model/xgboost_pipeline.py


In [13]:
%%writefile src/sarimax_model/train_predict_sarimax_model.py
"""
full pipeline for loading the data, training, and forecasting with SARIMAX.
"""
import polars as pl
import pandas as pd
import pmdarima as pm 
from sklearn.metrics import root_mean_squared_error
from datetime import datetime, timedelta
from typing import Tuple

from src.load_data import load_stocks
from src.sarimax_model.sarimax_etl import *
from src.model_preprocess import split_ar_on_cutoff
from src.utils import build_forecast_dates


def fit_sarimax(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    horizon_days: int,
    eval_mode: bool = True
):
    """fit the actual SARIMAX model.

    has two modes: eval and forecast. eval mode uses the train-eval split to 
    gauge how accurate the forecasts are (using RMSE). forecast mode uses the
    full dataset to train and make a forecast.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        label (str): which value to predict. defaults to "close".
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.
        eval_mode (bool, optional): whether to run the model as an evaluation of
            performance or as a full forecast. defaults to True (i.e., evaluate 
            the model performance).

    Returns:
        _type_: output depends on `eval_mode`. returns a float (RMSE) if 
            `eval_mode` is True, or a table (date, predicted value) if 
            `eval_mode` is False.
    """
    stock = [ticker]

    df_raw = load_stocks(stock, start_date, end_date)
    df_idx = build_df_with_indices(df_raw, label, start_date, end_date)

    feats = [
        "date",
        "dow",
        "month",
        "mon_or_fri",
        "volume",
        "SPY_close",
        "SPY_volume",
        "QQQ_close",
        "QQQ_volume",
        "IWM_close",
        "IWM_volume",
        "VXX_close",
        "VXX_volume",
        "UUP_close",
        "UUP_volume",
        "HYG_close",
        "HYG_volume",
        "LQD_close",
        "LQD_volume"
    ]

    exog_cols = [feat for feat in feats if feat != "date"]

    _sarima_hyperparams = {
        "start_p": 1,
        "start_q": 1,
        "test": "adf",
        "max_p": 3,
        "max_q": 3,
        "m": 5,
        "start_P": 0,
        "seasonal": True,
        "d": None,
        "D": None,
        "trace": False,
        "error_action": "ignore",
        "suppress_warnings": True,
        "stepwise": True
    }
    
    if eval_mode:
        df_train, df_eval = split_ar_on_cutoff(df_idx, cutoff, "close", feats)

        sarimax_model = pm.auto_arima(
            df_train[[label]],
            exogenous=df_train[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=len(df_eval),
            return_conf_int=True,
            exogenous=df_eval[exog_cols]
        )

        fitted = pd.DataFrame(fitted, columns=["pred"]).reset_index(drop=True)
        df_eval["pred"] = fitted["pred"]
        rmse = root_mean_squared_error(df_eval[["close"]], df_eval[["pred"]])
        
        return rmse
    else:
        df = df_idx.to_pandas()

        sarimax_model = pm.auto_arima(
            df[[label]],
            exogenous=df[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=horizon_days,
            return_conf_int=True,
            exogenous=df[exog_cols]
        )
        fitted = pd.DataFrame(fitted, columns=[f"pred_{label}"]).reset_index(
            drop=True
        )
        ci_series = pd.DataFrame(
            confint, columns=["lower_bound", "upper_bound"]
        )
        
        df_out = build_forecast_dates(
            end_date, horizon_days, skip_weekends=True
        )
        
        df_out[f"pred_{label}"] = fitted[f"pred_{label}"]
        df_out["lower_bound"] = ci_series["lower_bound"]
        df_out["upper_bound"] = ci_series["upper_bound"]

        return df_out.sort_values(by="date", ascending=True)

def sarimax_wrapper(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    horizon_days: int
) -> Tuple[pl.DataFrame, float]:
    """wrapper to run both versions of `fit_sarimax`.

    gets training results (`eval_mode == True`) and forecast results (`eval_mode
    == False`).

    Args:
        ticker (str): stock ticker to predict
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        label (str): which value to predict. defaults to "close".
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.

    Returns:
        Tuple[pl.DataFrame, float]: table of predictions (date, predicted value)
            and the RMSE from training.
    """
    rmse = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        horizon_days,
        eval_mode=True
    )

    forecasts = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        horizon_days,
        eval_mode=False
    )

    return pl.from_pandas(forecasts), rmse

Overwriting src/sarimax_model/train_predict_sarimax_model.py


In [14]:
%%writefile src/prophet_model/prophet_model_pipeline.py
"""_summary_

Returns:
    _type_: _description_
"""
import pandas as pd 
import polars as pl 
from prophet import Prophet 
from datetime import datetime
from typing import Tuple
from sklearn.metrics import root_mean_squared_error

import warnings 
warnings.filterwarnings("ignore")

from src.prophet_model.prophet_etl import build_prophet_df
from src.utils import build_forecast_dates
from src.model_preprocess import split_prophet_df


class ProphetForecaster:
    """wrapper class to train and forecast Prophet model."""

    def __init__(self, base_params: dict | None = None):
        self.base_params = base_params or {"interval_width": 0.95}
        self.model = None
        self.reg_cols = [] 
    
    def train_model(
        self,
        ticker: str,
        start_date: str,
        end_date: str,
        cutoff: datetime,
        label: str = "close",
        indexes: list[str] | None = None
    ) -> Tuple[float, pd.DataFrame]:
        """build data, train and evaluate Prophet model.

        data includes the stock label (defaults to close), index fund values,
        returns, RSI (7, 14, 21 days), and lag features of label (1, 7, 30 days)
        to predict future label values.

        Args:
            stock (list): stock to be predicted.
            start_date (str): start of historical price data.
            end_date (str): end of historical price data.
            cutoff (datetime): date to split for train/eval.
            label (str, optional): value to predict. defaults to "close".
            indexes (list[str] | None, optional): specific index funds to add to
                feature space. defaults to None.

        Returns:
            Tuple[float, pd.DataFrame]: RMSE value for evaluation and eval
                dataframe.
        """
        stock = [ticker]
        
        df = build_prophet_df(stock, start_date, end_date, label, indexes)
        df_train, df_eval = split_prophet_df(df, cutoff)

        self.reg_cols = [c for c in df.columns if c not in ["ds", "y"]]

        pr_model = Prophet(**self.base_params)
        for col in self.reg_cols:
            pr_model.add_regressor(col)

        pr_model.fit(df_train[["ds", "y"] + self.reg_cols])
        self.model = pr_model

        df_eval_future = df_eval[["ds"] + self.reg_cols].copy()
        forecast_eval = self.model.predict(df_eval_future)

        df_forecast = forecast_eval[["ds", "yhat", "yhat_lower", "yhat_upper"]]
        df_eval_out = df_eval[["ds", "y"]]
        df_eval_out = df_eval_out.merge(df_forecast, on="ds", how="inner")

        rmse = root_mean_squared_error(df_eval["y"], df_eval_out["yhat"])
        
        return rmse, df_eval_out
    
    def _build_future_regressors(
            self, df_full: pd.DataFrame, future_dates: pd.DataFrame
        ) -> pd.DataFrame:
        """stand-in function to make a dataframe for future predictions.

        will need to be added to if this were to ever go live for the sake of
        adding future regressors (and not just dates).

        Args:
            df_full (pd.DataFrame): full features dataframe.
            future_dates (pd.DataFrame): dataframe of days in the future.

        Returns:
            pd.DataFrame: single row containing date and regressors for making
                the forecast.
        """
        last_row = df_full.sort_values("ds").iloc[-1]
        date_list = future_dates["date"].tolist()
        
        rows = []

        for d in date_list:
            row = {"ds": d}
            for col in self.reg_cols:
                row[col] = last_row[col]
            rows.append(row)
        
        return pd.DataFrame(rows)
    
    def make_prediction(
        self,
        df_full: pd.DataFrame,
        horizon_days: int,
        label: str = "close"
    ) -> pl.DataFrame:
        """generate predictions `horizon_days` in the future.

        returns a polars dataframe for in-line displays.

        Args:
            df_full (pd.DataFrame): full features dataframe.
            horizon_days (int): number of days to forecast into the future.
            label (str, optional): specific value to forecast. defaults to 
            "close".

        Returns:
            pl.DataFrame: polars dataframe containing future dates, predicted
                values, and upper/lower bound CIs (95%).
        """
        assert self.model is not None

        last_date = df_full["ds"].max()

        future_dates = build_forecast_dates(
            last_date,
            horizon_days=horizon_days,
            skip_weekends=True
        )

        future_regs = self._build_future_regressors(df_full, future_dates)

        future_df = future_regs[["ds"] + self.reg_cols]

        forecast = self.model.predict(future_df)
        out = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
        df_out = pl.from_pandas(out)

        return df_out.select(
            pl.col("ds").alias("date"),
            pl.col("yhat").alias(f"pred_{label}"),
            pl.col("yhat_lower").alias("lower_bound"),
            pl.col("yhat_upper").alias("upper_bound")
        )

Overwriting src/prophet_model/prophet_model_pipeline.py


In [16]:
%%writefile src/utils.py
"""
utility functions
    build_forecast_dates
"""
from __future__ import annotations

from datetime import timedelta, date, datetime
import pandas as pd
from typing import List, Union
import numpy as np
import math

def build_forecast_dates(last_date, horizon_days: int) -> pd.DatetimeIndex:
    """return the next `horizon_days` trading dates (mon-fri), starting AFTER `last_date`.

    Args:
        last_date (_type_): final date of the training window.
        horizon_days (int): number of days for the forecast to project.

    Returns:
        pd.DatetimeIndex: index of datetimes for the forecast.
    """
    if isinstance(last_date, str):
        last_date = pd.to_datetime(last_date)
    
    dates = []
    d = pd.to_datetime(last_date)

    while len(dates) < horizon_days:
        d = d + pd.Timedelta(days=1)
        if d.weekday() < 5:
            dates.append(d)
    
    return pd.DatetimeIndex(dates)


def next_trading_day(d: datetime) -> datetime:
    """advance to next weekday (mon-fri)"""
    d = d + timedelta(days=1)
    while d.weekday() >= 5:
        d = d + pd.Timedelta(days=1)
    return d


def build_trading_future_dates(
    last_date: datetime, n_trading_days: int
) -> List[datetime]:
    """return next N trading dates strictly after `last_date`"""
    out = []
    d = last_date
    for _ in range(n_trading_days):
        d = next_trading_day(d)
        out.append(d)
    return out


def trading_days_between(start: date, end: date) -> int:
    """count trading days from start (exclusive) to end (inclusive)"""
    if end <= start:
        return 0
    
    days = 0
    cur = start 
    while cur < end:
        cur += timedelta(days=1)
        if cur.weekday() < 5:
            days += 1
    return days


def normal_params_from_quantiles(
    q10: float, q50: float, q90: float
) -> List[float]:
    """
    for a normal: q(p) = mu + z_p * sigma
    z_0.10 = -1.28155, z_0.50 = 0, z_0.90 = +1.28155
    """
    z = 1.2815515655446004
    mu = q50
    sigma = (q90 - q10) / (2 * z)
    sigma = max(sigma, 1e-9)
    return mu, sigma

Overwriting src/utils.py


In [10]:
%%writefile src/xgboost_model/xgboost_scorers.py
"""
collection of functions used to score the predictions from the trained xgboost
options models.
"""
import pandas as pd
import polars as pl
import numpy as np
from math import log, sqrt, exp 
from scipy.stats import norm 
from datetime import date 
from typing import Dict, Tuple

from src.load_data import get_option_chain, list_expiries, pick_expiries_for_horizons
from src.utils import normal_params_from_quantiles, trading_days_between

def simulate_terminal_prices(
    spot: float,
    mu: float,
    sigma: float,
    T_years: float,
    n_sims: int = 50_000,
    seed: int = 7
) -> np.ndarray:
    """simulate terminal underlying prices under a lognormal return assumption.

    Args:
        spot (float): current underlying spot price.
        mu (float): mean of the forecast return distribution (log-return).
        sigma (float): SD of the forecast return distribution
        T_years (float): time to expiration (years).
        n_sims (int, optional): number of monte carlo simulations. defaults to 
            50_000.
        seed (int, optional): random seed. defaults to 7.

    Returns:
        np.ndarray: simulated terminal price array of shape (n_sims).
    """
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n_sims)
    r = mu + sigma * z
    ST = spot * np.exp(r)
    return ST 


def payoff_call(ST: np.ndarray, K: float) -> np.ndarray:
    """compute payoff of long call option at expiration."""
    return np.maximum(ST - K, 0.0)


def payoff_put(ST: np.ndarray, K: float) -> np.ndarray:
    """compute payoff of long put option at expiration."""
    return np.maximum(K - ST, 0.0)


def payoff_vertical_call_spread(
    ST: np.ndarray, K_long: float, K_short: float
) -> np.ndarray:
    """compute payoff of a bull call vertical spread at expiration."""
    return payoff_call(ST, K_long) - payoff_call(ST, K_short)


def payoff_vertical_put_spread(
    ST: np.ndarray, K_short: float, K_long: float
) -> np.ndarray:
    """compute payoff of a bear put vertical spread at expiration."""
    return payoff_put(ST, K_long) - payoff_put(ST, K_short)


def bs_d1_d2(
    S: float, K: float, T: float, r: float, q: float, iv: float
) -> Tuple[float, float]:
    """compute black-scholes d1 and d2 parameters.

    Args:
        S (float): spot price.
        K (float): strike price.
        T (float): time to expiration (years).
        r (float): risk-free rate.
        q (float): dividend yield.
        iv (float): implied volatility.

    Returns:
        Tuple[float, float]: (d1, d2) parameters used in BS pricing.
    """
    if T <= 0 or iv <= 0:
        return np.nan, np.nan 
    
    d1 = (log(S/K) + (r - q + 0.5*iv*iv) * T) / (iv*sqrt(T))
    d2 = d1 - iv*sqrt(T)
    return d1, d2


def bs_price(
    S: float, 
    K: float, 
    T: float, 
    r: float = 0.04, 
    q: float = 0.0, 
    iv: float = 0.8, 
    opt_type: str = "call"
) -> float:
    """compute black-scholes theoretical option price.

    Args:
        S (float): spot price.
        K (float): strike price.
        T (float): time to expiration in years.
        r (float, optional): risk-free rate. defaults to 0.04.
        q (float, optional): dividend yield. defaults to 0.0.
        iv (float, optional): implied volatility. defaults to 0.8.
        opt_type (str, optional): 'call' or 'put'. defaults to 'call'.

    Returns:
        float: theoretical black-scholes option price.
    """
    d1, d2 = bs_d1_d2(S, K, T, r, q, iv)
    if np.isnan(d1):
        return np.nan 
    if opt_type == "call":
        return S * exp(-q*T) * norm.cdf(d1) - K * exp(-r*T) * norm.cdf(d2)
    else:
        return K * exp(-r*T) * norm.cdf(-d2) - S * exp(-q*T) * norm.cdf(-d1)
    

def bs_prob_itm(
    S: float, 
    K: float, 
    T: float, 
    r: float = 0.04, 
    q: float = 0.0, 
    iv: float = 0.8, 
    opt_type: str = "call"
) -> float:
    """estimate risk-neutral probability of finishing ITM under black-scholes.

    under BS risk-neutral, P(call ITM) ~ N(d2), P(put ITM) ~ N(-d2)
    
    Args:
        S (float): spot price.
        K (float): strike price.
        T (float): time to expiration in years.
        r (float, optional): risk-free rate. defaults to 0.04.
        q (float, optional): dividend yield. defaults to 0.0.
        iv (float, optional): implied volatility. defaults to 0.8.
        opt_type (str, optional): 'call' or 'put'. defaults to 'call'.

    Returns:
        float: approximate risk-neutral probability of expiring ITM.
    """
    d1, d2 = bs_d1_d2(S, K, T, r, q, iv)
    if np.isnan(d2):
        return np.nan 
    return float(norm.cdf(d2) if opt_type == "call" else norm.cdf(-d2))


def mid_price(row: pd.Series) -> float:
    """compute midpoint price from bid/ask with fallback to last trade."""
    b, a = row.get("bid", np.nan), row.get("ask", np.nan)
    if pd.notna(b) and pd.notna(a) and a > 0:
        return float((b + a) / 2)
    
    last = row.get("lastPrice", np.nan)
    return float(last) if pd.notna(last) else np.nan 


def score_single_leg(
    chain: pd.DataFrame,
    spot: float,
    mu: float,
    sigma: float,
    T_years: float,
    opt_type: str,
    r: float = 0.04,
    q: float = 0.0,
    n_sims: int = 50_000
) -> pd.DataFrame:
    """score single-leg options using monte carlo under model-implied distribution.

    simulates terminal prices, computes payoff and PnL for each strike, and
    estimates probability of ITM, probability of profit, and expected value.
    also compares model-implied ITM probability to IV-implied probability.

    Args:
        chain (pd.DataFrame): option chain (calls or puts).
        spot (float): current spot price.
        mu (float): forecast mean return.
        sigma (float): forecast return volatility.
        T_years (float): time to expiration in years.
        opt_type (str): 'call' or 'put'.
        r (float, optional): risk-free rate. defaults to 0.04.
        q (float, optional): eividend yield. defaults to 0.0.
        n_sims (int, optional): monte carlo simulations. defaults to 50_000.

    Returns:
        pd.DataFrame: scored options sorted by EV and probability of profit.
    """
    ST = simulate_terminal_prices(spot, mu, sigma, T_years, n_sims=n_sims)

    rows = []
    for _, row in chain.iterrows():
        K = float(row["strike"])
        premium = mid_price(row)
        if not np.isfinite(premium) or premium <= 0:
            continue 
        
        if opt_type == "call":
            payoff = payoff_call(ST, K)
        else:
            payoff = payoff_put(ST, K)
        
        pnl = payoff - premium

        p_itm_model = float((payoff > 0).mean())
        pop_model = float((pnl > 0).mean())
        ev_model = float(pnl.mean())

        iv = row.get("impliedVolatility", np.nan)
        p_itm_mkt = (
            bs_prob_itm(
                spot, K, T_years, r=r, q=q, iv=float(iv), opt_type=opt_type
            )
            if pd.notna(iv) and float(iv) > 0
            else np.nan 
        )

        rows.append(
            {
                "type": opt_type,
                "strike": K,
                "premium_mid": premium,
                "iv": float(iv) if pd.notna(iv) else np.nan,
                "p_itm_model": p_itm_model,
                "p_itm_mkt_iv": p_itm_mkt,
                "edge_itm": (p_itm_model - p_itm_mkt) if np.isfinite(p_itm_mkt) else np.nan,
                "pop_model": pop_model,
                "ev_model_per_share": ev_model,
                "ev_model_per_contract": ev_model * 100.0
            }
        )
    
    df_out = pd.DataFrame(rows)
    if len(df_out) == 0:
        return df_out
    
    return df_out.sort_values(
        ["ev_model_per_contract", "pop_model"], ascending=False
    )


def score_vertical_call_spreads(
    calls: pd.DataFrame,
    spot: float,
    mu: float,
    sigma: float,
    T_years: float,
    n_sims: int = 50_000,
    max_legs: int = 40
) -> pd.DataFrame:
    """score bull call spreads using monte carlo under forecast distribution.

    constructs candidate long/short call combinations and estimates expected
    value, probability of profit, max profit, and max loss.

    Args:
        calls (pd.DataFrame): call option chain.
        spot (float): current spot price.
        mu (float): forecast mean return.
        sigma (float): forecast return volatility.
        T_years (float): time to expiration in years.
        n_sims (int, optional): monte carlo simulations. defaults to 50_000.
        max_legs (int, optional): limit on strikes evaluated. defaults to 40.

    Returns:
        pd.DataFrame: scored bull call spreads sorted by EV.
    """
    ST = simulate_terminal_prices(spot, mu, sigma, T_years, n_sims=n_sims)

    calls2 = calls.copy()
    calls2["premium_mid"] = calls2.apply(mid_price, axis=1)
    calls2 = calls2.dropna(subset=["premium_mid"])
    calls2 = calls2.sort_values("strike")
    calls2 = calls2.head(max_legs)

    strikes = calls2["strike"].to_numpy(float)
    prem = calls2["premium_mid"].to_numpy(float)

    rows = []
    for i in range(len(strikes)):
        for j in range(i + 1, len(strikes)):
            K_long, K_short = strikes[i], strikes[j]
            debit = prem[i] - prem[j]
            if debit <= 0:
                continue 

            payoff = payoff_vertical_call_spread(ST, K_long, K_short)
            pnl = payoff - debit 
            rows.append(
                {
                    "spread": "bull_call",
                    "K_long": float(K_long),
                    "K_short": float(K_short),
                    "debit_mid": float(debit),
                    "pop_model": float((pnl > 0).mean()),
                    "ev_per_share": float(pnl.mean()),
                    "ev_per_contract": float(pnl.mean() * 100.0),
                    "max_profit": float((K_short - K_long - debit) * 100.0),
                    "max_loss": float(debit * 100.0)
                }
            )
    
    df_out = pd.DataFrame(rows)
    if df_out.empty:
        return pd.DataFrame(
            columns=[
                "spread",
                "K_long",
                "K_short",
                "debit_mid",
                "pop_model",
                "ev_per_share",
                "ev_per_contract",
                "max_profit",
                "max_loss",
            ]
        )
    return df_out.sort_values(["ev_per_contract", "pop_model"], ascending=False)


def score_vertical_put_spreads(
    puts: pd.DataFrame,
    spot: float,
    mu: float,
    sigma: float,
    T_years: float,
    n_sims: int = 50_000,
    max_legs: int = 40
) -> pd.DataFrame:
    """score bear put spreads using monte carlo under forecast distribution.

    constructs candidate long/short call combinations and estimates expected
    value, probability of profit, max profit, and max loss.

    Args:
        calls (pd.DataFrame): call option chain.
        spot (float): current spot price.
        mu (float): forecast mean return.
        sigma (float): forecast return volatility.
        T_years (float): time to expiration in years.
        n_sims (int, optional): monte carlo simulations. defaults to 50_000.
        max_legs (int, optional): limit on strikes evaluated. defaults to 40.

    Returns:
        pd.DataFrame: scored bear put spreads sorted by EV.
    """
    ST = simulate_terminal_prices(spot, mu, sigma, T_years, n_sims=n_sims)

    puts2 = puts.copy()
    puts2["premium_mid"] = puts2.apply(mid_price, axis=1)
    puts2 = puts2.dropna(subset=["premium_mid"])
    puts2 = puts2.sort_values("strike")
    puts2 = puts2.head(max_legs)

    strikes = puts2["strike"].to_numpy(float)
    prem = puts2["premium_mid"].to_numpy(float)

    rows = []
    for i in range(len(strikes)):
        for j in range(i + 1, len(strikes)):
            K_short, K_long = strikes[i], strikes[j]
            if K_long <= K_short:
                continue 
            
            debit = prem[j] - prem[i]
            if debit <= 0:
                continue 

            payoff = payoff_vertical_put_spread(ST, K_short, K_long)
            pnl = payoff - debit 
            rows.append(
                {
                    "spread": "bear_put",
                    "K_long": float(K_long),
                    "K_short": float(K_short),
                    "debit_mid": float(debit),
                    "pop_model": float((pnl > 0).mean()),
                    "ev_per_share": float(pnl.mean()),
                    "ev_per_contract": float(pnl.mean() * 100.0),
                    "max_profit": float((K_long - K_short - debit) * 100.0),
                    "max_loss": float(debit * 100.0)
                }
            )
    
    df_out = pd.DataFrame(rows)
    if df_out.empty:
        return pd.DataFrame(
            columns=[
                "spread",
                "K_long",
                "K_short",
                "debit_mid",
                "pop_model",
                "ev_per_share",
                "ev_per_contract",
                "max_profit",
                "max_loss",
            ]
        )
    return df_out.sort_values(["ev_per_contract", "pop_model"], ascending=False)


def evaluate_options_across_expiries(
    ticker: str,
    quantile_forecasts: pl.DataFrame,
    r: float = 0.04,
    q: float = 0.0,
    n_sims: int = 50_000
) -> dict:
    """evaluate options across forecast horizons and nearest listed expiries.

    F\for each forecast horizon, maps to a listed expiration, derives the model-
    implied return distribution from quantiles, scores single-leg options and
    vertical spreads, and aggregates results into a structured dictionary.

    Args:
        ticker (str): underlying ticker symbol.
        quantile_forecasts (pl.DataFrame): forecast output containing horizon,
            spot, and return quantiles.
        r (float, optional): risk-free rate. defaults to 0.04.
        q (float, optional): dividend yield. defaults to 0.0.
        n_sims (int, optional): monte carlo simulations. defaults to 50_000.

    Returns:
        dict: nested dictionary keyed by horizon containing scored options.
    """
    quantile_forecasts = quantile_forecasts.to_pandas()
    asof = date.today()
    out = {}
    
    listed_expiries = list_expiries(ticker)
    if not listed_expiries:
        raise ValueError(
            f"no listed expirations found for ticker `{ticker}`..."
        )
    listed_expiry_dates = [pd.to_datetime(s).date() for s in listed_expiries]

    for _, row in quantile_forecasts.iterrows():
        spot = float(row["spot"])
        H = int(row["horizon"])
        
        forecast_expiry = pd.to_datetime(row["date"]).date()
        expiry_date = min(
            listed_expiry_dates, key=lambda d: abs((d - forecast_expiry).days)
        )
        T_days = trading_days_between(asof, expiry_date)
        T_years = max(T_days / 252.0, 1e-6)

        mu, sigma = normal_params_from_quantiles(
            float(row["pred_q10_ret"]),
            float(row["pred_q50_ret"]),
            float(row["pred_q90_ret"])
        )

        # expiry_str = expiry_date.strftime("%Y-%m-%d")
        expiry_map = pick_expiries_for_horizons(
            ticker, 
            sorted(quantile_forecasts["horizon"].unique().tolist()), 
            asof=date.today()
        )
        expiry_str = expiry_map[H]
        calls, puts = get_option_chain(ticker, expiry_str)

        scored_calls = score_single_leg(
            calls, spot, mu, sigma, T_years, opt_type="call", r=r, q=q, n_sims=n_sims
        )
        scored_puts = score_single_leg(
            puts, spot, mu, sigma, T_years, opt_type="put", r=r, q=q, n_sims=n_sims
        )

        call_spreads = score_vertical_call_spreads(
            calls, spot, mu, sigma, T_years, n_sims=n_sims
        )
        put_spreads = score_vertical_put_spreads(
            puts, spot, mu, sigma, T_years, n_sims=n_sims
        )

        out[H] = {
            "expiry": expiry_str,
            "mu": mu,
            "sigma": sigma,
            "spot": spot,
            "pred_q10_px": float(row.get("pred_q10_px", float("nan"))),
            "pred_q50_px": float(row.get("pred_q50_px", float("nan"))),
            "pred_q90_px": float(row.get("pred_q90_px", float("nan"))),
            "calls": scored_calls,
            "puts": scored_puts,
            "bull_call_spreads": call_spreads,
            "bear_put_spreads": put_spreads
        }
    
    return out


def _normalize_results(results: dict) -> dict:
    """normalize results dictionary keys to integers."""
    if isinstance(results, dict):
        return {int(k): v for k, v in results.items()}


def parse_options_results(
    results: dict, cols: list
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """flatten nested option scoring results into summary and detail tables.

    extracts summary statistics and concatenates calls, puts, and spread results
    across horizons into unified DataFrames.

    Args:
        results (dict): nested results from evaluate_options_across_expiries.
        cols (list): keys to extract for summary table.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
            summary, calls, puts, bull spreads, bear spreads.
    """
    r = _normalize_results(results)
    summary_rows = []
    calls_parts = []
    puts_parts = []
    bull_call_parts = []
    bear_put_parts = []

    for h, payload in sorted(r.items()):
        h = int(h)

        row = {"horizon": h}
        for c in cols:
            row[c] = payload.get(c)
        summary_rows.append(row)

        calls_df = payload.get("calls")
        puts_df = payload.get("puts")
        bull_df = payload.get("bull_call_spreads")
        bear_df = payload.get("bear_put_spreads")

        if isinstance(calls_df, pd.DataFrame) and not calls_df.empty:
            calls_parts.append(calls_df.assign(horizon=h))
        if isinstance(puts_df, pd.DataFrame) and not puts_df.empty:
            puts_parts.append(puts_df.assign(horizon=h))
        if isinstance(bull_df, pd.DataFrame) and not bull_df.empty:
            bull_call_parts.append(bull_df.assign(horizon=h))
        if isinstance(bear_df, pd.DataFrame) and not bear_df.empty:
            bear_put_parts.append(bear_df.assign(horizon=h))
        
    summary_df = pd.DataFrame(summary_rows, columns=["horizon", *cols])
    calls_out = pd.concat(calls_parts, ignore_index=True) if calls_parts else pd.DataFrame({"horizon": []})
    puts_out = pd.concat(puts_parts, ignore_index=True) if puts_parts else pd.DataFrame({"horizon": []})
    bull_calls_out = pd.concat(bull_call_parts, ignore_index=True) if bull_call_parts else pd.DataFrame({"horizon": []})
    bear_puts_out = pd.concat(bear_put_parts, ignore_index=True) if bear_put_parts else pd.DataFrame({"horizon": []})

    return summary_df, calls_out, puts_out, bull_calls_out, bear_puts_out


def option_score_single_leg(
    ev_model_per_contract: float, 
    premium_mid: float, 
    pop_model: float, 
    edge_itm: float
) -> float:
    """compute composite score for ranking single-leg options."""
    edge_wt = 1 / (1 + np.exp(-edge_itm * 10))
    return (ev_model_per_contract / (premium_mid * 100)) * pop_model * edge_wt


def make_trade_gates_single_leg(
    ev_model_per_contract: float, 
    edge_itm: float, 
    pop_model: float,
    empc_threshold: int = 10,
    ei_threshold: float = 0.03,
    pop_threshold: float = 0.45,
) -> float:
    """evaluate trade gating criteria for single-leg options."""
    passes = 0
    if ev_model_per_contract > empc_threshold:
        passes += 1
    if edge_itm > ei_threshold:
        passes += 1
    if pop_model > pop_threshold:
        passes += 1
    
    return passes / 3


def option_score_spread_roi(ev_per_contract: float, max_loss: float) -> float:
    """compute expected return on risk for a vertical spread."""
    return ev_per_contract / max_loss 


def option_score_spread_efficiency(
    ev_per_contract: float, max_loss: float, pop_model: float
) -> float:
    """compute risk-adjuted efficiency score for a spread."""
    return (ev_per_contract / max_loss) * pop_model 


def option_score_spread_max_ratio(max_profit: float, max_loss: float) -> float:
    """compute reward-to-risk ratio for a spread."""
    return max_profit / max_loss 


def score_options_results(
    results: dict,
    empc_threshold: int = 10,
    ei_threshold: float = 0.03,
    pop_threshold: float = 0.45,
    mr_threshold: float = 0.50,
    loss_threshold: int = 100
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """aggregate and rank scored options into decision-ready tables.

    applies trade gates and scoring functions to single-leg and spread
    results, filters by risk constraints, and returns top candidates
    per horizon and structure.

    Args:
        results (dict): nested output from evaluate_options_across_expiries.
        empc_threshold (int, optional): EV threshold for single legs.
        ei_threshold (float, optional): edge threshold for single legs.
        pop_threshold (float, optional): probability threshold.
        mr_threshold (float, optional): minimum reward-to-risk for spreads.
        loss_threshold (int, optional):maximum allowed risk per spread.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
            summary table, top single-leg candidates, top spread candidates.
    """
    cols_list = [
        "expiry",
        "mu",
        "sigma",
        "spot",
        "pred_q10_px",
        "pred_q50_px",
        "pred_q90_px"
    ]

    summary_res, call_res, put_res, bull_res, bear_res = parse_options_results(
        results, cols_list
    )

    single_leg_res = pd.concat([call_res, put_res])

    single_leg_res.query(
        f"ev_model_per_contract > 0 & edge_itm > 0 & pop_model > {pop_threshold}",
        inplace=True
    )

    single_leg_res["option_score"] = single_leg_res.apply(
        lambda x: option_score_single_leg(
            x["ev_model_per_contract"], x["premium_mid"], x["pop_model"], x["edge_itm"]
        ), axis=1
    )

    single_leg_res["outcome_score"] = single_leg_res.apply(
        lambda x: make_trade_gates_single_leg(
            x["ev_model_per_contract"], x["edge_itm"], x["pop_model"]
        ), axis=1
    )

    spread_res = pd.concat([bull_res, bear_res])

    spread_res.query(
        f"ev_per_contract > 0 & pop_model > {pop_threshold}", inplace=True
    )

    spread_res["roi_ev"] = spread_res.apply(
        lambda x: option_score_spread_roi(x["ev_per_contract"], x["max_loss"]),
        axis=1
    )

    spread_res["efficiency_score"] = spread_res.apply(
        lambda x: option_score_spread_efficiency(
            x["ev_per_contract"], x["max_loss"], x["pop_model"]
        ), axis=1
    )

    spread_res["profit_ratio"] = spread_res.apply(
        lambda x: option_score_spread_max_ratio(
            x["max_profit"], x["max_loss"]
        ), axis=1
    )

    spread_res.query(
        f"profit_ratio >= {mr_threshold} & max_loss <= {loss_threshold}",
        inplace=True
    )

    return (
        summary_res,

        single_leg_res.sort_values(
            ["horizon", "option_score"],
            ascending=[True, False]
        ).groupby(
            ["horizon", "type"]
        ).head(10),

        spread_res.sort_values(
            ["horizon", "efficiency_score"], 
            ascending=[True, False]
        ).groupby(
            ["horizon", "spread"]
        ).head(10)
    )

Overwriting src/xgboost_model/xgboost_scorers.py


In [1]:
# %%writefile workflows/train_and_forecast_model.py
"""
script for getting all model results.
"""
import pandas as pd 
import polars as pl 
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from timeit import default_timer as timer

import warnings
warnings.filterwarnings("ignore")

from src.xgboost_model.xgboost_pipeline import train_and_forecast_xgb_options
from src.xgboost_model.xgboost_scorers import evaluate_options_across_expiries, score_options_results

ticker = "AAPL"
horizons = [10, 21, 40, 90, 180]

end_date = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - relativedelta(years=10)).strftime("%Y-%m-%d")

max_horizon = max(horizons)
if max_horizon > 180:
    cm = 18
elif max_horizon > 90:
    cm = 12
elif max_horizon > 40:
    cm = 6
else:
    cm = 2

print(
    f"\n\nmax horizon is {max_horizon} days, "
    f"setting cutoff to {cm} months for train-test split."
)

cutoff = (datetime.today() - relativedelta(months=cm))
print("")
label = "close"

print(
    f"forecasting '{ticker}' \033[4m{label}\033[0m prices "
    f"over the next {horizons} days"
)
print(
    f"training models from '{start_date}' to '{end_date}', "
    f"splitting on '{cutoff.strftime("%Y-%m-%d")}'\n\n"
)

timer_xgb_start = timer()

xgb_forecasts, xgb_rmse, xgb_improvements = train_and_forecast_xgb_options(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    cutoff=cutoff,
    label_col=label,
    horizons=horizons,
    quantiles=[0.1, 0.5, 0.9],
    label_mode="log_return",
    n_estimators=400,
    learning_rate=0.03
)
timer_xgb_end = timer() - timer_xgb_start
print(f"XGBoost run duration: {timer_xgb_end:.5f} seconds\n\n")

with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=10000):
    print(xgb_forecasts)
    print(xgb_rmse)
    print(xgb_improvements)

results = evaluate_options_across_expiries(
    ticker=ticker,
    quantile_forecasts=xgb_forecasts,
    r=0.04,
    q=0.0,
    n_sims=50_000
)

summary_res, single_leg_res, spread_res = score_options_results(results)



max horizon is 180 days, setting cutoff to 12 months for train-test split.

forecasting 'AAPL' close prices over the next [10, 21, 40, 90, 180] days
training models from '2016-02-23' to '2026-02-23', splitting on '2025-02-23'




[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


XGBoost run duration: 11.84643 seconds


shape: (5, 9)
┌───────────┬──────────┬─────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ date      ┆ spot     ┆ horizon ┆ pred_q10 ┆ pred_q10 ┆ pred_q50 ┆ pred_q50 ┆ pred_q90 ┆ pred_q90 │
│ ---       ┆ ---      ┆ ---     ┆ _ret     ┆ _px      ┆ _ret     ┆ _px      ┆ _ret     ┆ _px      │
│ datetime[ ┆ f64      ┆ i64     ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ ns]       ┆          ┆         ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞═══════════╪══════════╪═════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ 2026-03-0 ┆ 264.5799 ┆ 10      ┆ -0.06737 ┆ 247.3404 ┆ -0.01355 ┆ 261.0190 ┆ 0.037774 ┆ 274.7653 │
│ 6         ┆ 87       ┆         ┆ 8        ┆ 44       ┆          ┆ 63       ┆          ┆ 67       │
│ 00:00:00  ┆          ┆         ┆          ┆          ┆          ┆          ┆          ┆          │
│ 2026-03-2 ┆ 264.5799 ┆ 21      ┆ -

In [ ]:
# import numpy as np
# y_true = df_eval["label_return"].values
# rmse_zero = np.sqrt(np.mean((y_true - 0.0)**2))
# rmse_model = np.sqrt(np.mean((y_true - y_pred)**2))
# improvement = 1 - (rmse_model / rmse_zero)

# rmse_zero, rmse_model, improvement

In [2]:
summary_res

,horizon,expiry,mu,sigma,spot,pred_q10_px,pred_q50_px,pred_q90_px
0,10,2026-03-06,-0.028208,0.038161,264.579987,249.003684,257.220945,274.589784
1,21,2026-03-20,-0.084413,0.029515,264.579987,246.534104,243.162581,265.908230
2,40,2026-04-17,-0.139884,0.052072,264.579987,226.486794,230.041437,258.825395
3,90,2026-06-18,-0.167695,0.067564,264.579987,220.119838,223.731937,261.738388
4,180,2026-11-20,0.135923,0.033341,264.579987,276.995269,303.101264,301.707133


In [3]:
single_leg_res

,type,strike,premium_mid,iv,p_itm_model,p_itm_mkt_iv,edge_itm,pop_model,ev_model_per_share,ev_model_per_contract,horizon,option_score,outcome_score
8,put,270.0,7.850,0.236458,0.89950,0.662889,0.236611,0.69360,5.285255,528.525506,10,0.426923,1.0
9,put,265.0,5.150,0.252693,0.78346,0.510031,0.273429,0.60748,3.753403,375.340341,10,0.415743,1.0
7,put,275.0,11.400,0.221321,0.96172,0.805706,0.156014,0.74076,6.405851,640.585130,10,0.343974,1.0
10,put,260.0,3.275,0.269905,0.61312,0.371677,0.241443,0.48174,2.114064,211.406400,10,0.285447,1.0
6,put,280.0,15.775,0.279060,0.98806,0.845717,0.142343,0.76072,6.915343,691.534337,10,0.268743,1.0
2,put,285.0,20.500,0.345221,0.99688,0.862668,0.134212,0.76918,7.156653,715.665286,10,0.212897,1.0
1,put,290.0,25.425,0.404425,0.99930,0.876828,0.122472,0.77128,7.223687,722.368672,10,0.169367,1.0
0,put,295.0,30.325,0.471685,0.99986,0.882651,0.117209,0.77390,7.321975,732.197471,10,0.142671,1.0
3,put,300.0,35.600,0.576176,0.99998,0.872480,0.127500,0.76618,7.046689,704.668858,10,0.118536,1.0
4,put,305.0,40.625,0.627201,1.00000,0.882513,0.117487,0.76538,7.021682,702.168168,10,0.101072,1.0


In [4]:
spread_res

,spread,K_long,K_short,debit_mid,pop_model,ev_per_share,ev_per_contract,max_profit,max_loss,horizon,roi_ev,efficiency_score,profit_ratio
875,bear_put,250.0,245.0,0.765,0.79926,2.851224,285.122387,423.5,76.5,21,3.727090,2.978914,5.535948
909,bear_put,245.0,240.0,0.515,0.57618,1.812128,181.212808,448.5,51.5,21,3.518695,2.027402,8.708738
877,bear_put,245.0,235.0,0.890,0.55580,2.523274,252.327364,911.0,89.0,21,2.835139,1.575770,10.235955
1759,bear_put,245.0,240.0,0.800,0.87594,3.419147,341.914749,420.0,80.0,40,4.273934,3.743710,5.250000
1768,bear_put,240.0,235.0,0.675,0.77724,2.976291,297.629096,432.5,67.5,40,4.409320,3.427100,6.407407
1790,bear_put,235.0,230.0,0.485,0.64786,2.430682,243.068213,451.5,48.5,40,5.011716,3.246890,9.309278
1754,bear_put,235.0,225.0,0.870,0.63556,4.130779,413.077862,913.0,87.0,40,4.748021,3.017652,10.494253
1797,bear_put,230.0,225.0,0.385,0.48772,1.700096,170.009648,461.5,38.5,40,4.415835,2.153691,11.987013
1786,bear_put,230.0,220.0,0.690,0.47682,2.706983,270.698260,931.0,69.0,40,3.923163,1.870643,13.492754
1764,bear_put,230.0,215.0,0.930,0.46886,3.183920,318.392046,1407.0,93.0,40,3.423570,1.605175,15.129032


In [ ]:
## save for later
# from src.sarimax_model.train_predict_sarimax_model import *
# from src.prophet_model.prophet_model_pipeline import ProphetForecaster
# from src.prophet_model.prophet_etl import build_prophet_df



# timer_smax_start = timer()

# smax_forecasts, smax_rmse = sarimax_wrapper(
#     ticker=ticker,
#     start_date=start_date,
#     end_date=end_date,
#     label=label,
#     cutoff=cutoff,
#     horizon_days=horizon,    
# )
# timer_smax_end = timer() - timer_smax_start

# print(f"\nSARIMAX test RMSE on holdout: {smax_rmse:.4f}\n")
# print(f"SARIMAX run duration: {timer_smax_end:.5f} seconds\n\n")
# print(smax_forecasts)
# print("\n\n")


# timer_prph_start = timer()

# idxs = ["VXX", "QQQ", "SPY", "IWM"]

# forecaster = ProphetForecaster()
# prph_rmse, eval_df = forecaster.train_model(
#     ticker=ticker,
#     start_date=start_date,
#     end_date=end_date,
#     cutoff=cutoff,
#     label=label,
#     indexes=idxs
# )
# df_full = build_prophet_df(
#     stocks=[ticker], start=start_date, end=end_date, label=label, indexes=idxs
# )
# prph_forecasts = forecaster.make_prediction(
#     df_full=df_full, horizon_days=horizon
# )
# timer_prph_end = timer() - timer_prph_start

# print(f"\nProphet test RMSE on holdout: {prph_rmse:.4f}\n")
# print(f"Prophet run duration: {timer_prph_end:.5f} seconds\n\n")
# print(prph_forecasts)
# print("\n\n")